# 🫀 Heart Disease Prediction Hackathon — สรุปทุกขั้นตอนแบบละเอียด
ภาพรวมโจทย์
เป้าหมาย: ทำนายว่าผู้ป่วยแต่ละคน เคยเป็นโรคหัวใจหรือไม่ (Yes / No) โดยอ่านจากข้อมูลสุขภาพ เช่น BMI, ความดัน, รายได้, การศึกษา ฯลฯ

**ข้อจำกัด:** ห้ามใช้โมเดลภายนอก (Pre-trained Model) ต้องเทรนเองจากข้อมูลที่ให้มาเท่านั้น
ไฟล์ที่ได้รับ:
train.csv — ข้อมูลสำหรับสอนโมเดล (มีคำตอบ Yes/No ให้ดูเป็นตัวอย่าง)
test.csv — ข้อมูลที่ต้องทำนาย (ไม่มีคำตอบ)
sample_submission.csv — ตัวอย่างรูปแบบไฟล์ที่ต้องส่ง Kaggle

In [40]:
!pip install xgboost
!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 286.5 kB/s  0:00:06271.0 kB/s eta 0:00:02


Import Libary

In [35]:
import pandas as pd
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# 1. โหลดข้อมูลทำเป็น Dataframe ด้วย pandas library
train_df = pd.read_csv("heart_prediction_data/train.csv")
test_df = pd.read_csv("heart_prediction_data/test.csv")
print("จำนวนแถว Train:", len(train_df), "| ตรวจสอบค่าว่าง (โชว์แค่คอลัมน์ที่ไม่ปกติ):")
print(train_df.isna().sum()[train_df.isna().sum() > 0])
# ตรวจสอบของ test set ด้วยเพราะเราต้องทำทุกกระบวนการเหมือนกันที่ทำกับ trainset 
print("จำนวนแถว Train:", len(test_df), "| ตรวจสอบค่าว่าง (โชว์แค่คอลัมน์ที่ไม่ปกติ):")
print(test_df.isna().sum()[test_df.isna().sum() > 0])

จำนวนแถว Train: 223084 | ตรวจสอบค่าว่าง (โชว์แค่คอลัมน์ที่ไม่ปกติ):
History of HeartDisease or Attack     1694
Told High Cholesterol                32186
Body Mass Index                      11782
Smoked 100+ Cigarettes                   1
Diagnosed Diabetes                       3
Doctor Visit Cost Barrier                1
General Health                           1
Difficulty Walking                       3
dtype: int64
จำนวนแถว Train: 74361 | ตรวจสอบค่าว่าง (โชว์แค่คอลัมน์ที่ไม่ปกติ):
Series([], dtype: int64)


In [47]:
display(train_df)

,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,train_000002,No,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,train_000004,No,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
4,train_000005,NaN,Yes,Yes,Yes,34.56,Yes,No,No,Yes,No,Yes,Yes,Very Poor,Yes,Male,Some high school,"$15,000 to less than $20,000",98,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223079,train_223080,No,No,No,Yes,28.20,No,No,No,No,No,Yes,No,Excellent,No,Female,College graduate,"$25,000 to less than $35,000",34,Yes
223080,train_223081,No,Yes,Yes,Yes,45.01,No,No,No,No,No,Yes,No,Fair,No,Male,College graduate,"$50,000 to less than $75,000",43,Yes
223081,train_223082,No,Yes,Yes,Yes,18.94,No,No,Yes,No,No,Yes,No,Poor,Yes,Female,Elementary,"$20,000 to less than $25,000",72,No
223082,train_223083,No,No,No,Yes,29.29,No,No,No,Yes,No,Yes,No,Excellent,No,Female,Some college or technical school,"($10,000 to less than $15,000",28,Yes


**จากการแสดงผลด้านบน จะเห็นได้ว่าข้อมูล ของ trainset ยังไม่สะอาดเราต้องทำการจัดการความผิดปกติในข้อมูลออก**

In [48]:
from sklearn.preprocessing import LabelEncoder

# 2. จัดการค่าว่าง (Missing Values)
for col in train_df.columns:
    if train_df[col].dtype == 'object':
        train_df[col].fillna(train_df[col].mode()[0], inplace=True)
        if col in test_df.columns:
            test_df[col].fillna(train_df[col].mode()[0], inplace=True)
    else:
        train_df[col].fillna(train_df[col].median(), inplace=True)
        if col in test_df.columns:
            test_df[col].fillna(train_df[col].median(), inplace=True)

# 3. จัดการ Outliers ในหมวดตัวเลข
def clip_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[column] = np.clip(df[column], a_min=lower_bound, a_max=upper_bound)
    return df

train_df = clip_outliers(train_df, 'Body Mass Index')
test_df = clip_outliers(test_df, 'Body Mass Index')


# ==========================================
# 💎 3.5 แปลงข้อความที่เรียงลำดับได้ (Ordinal Mapping - อัปเกรด Feature)
# บังคับเรียงลำดับขั้นตามชีวิตจริง เพื่อให้โมเดลแยกแยะง่ายขึ้น
# ==========================================

# สุขภาพ
health_map = {'Very Poor': 1, 'Poor': 2, 'Fair': 3, 'Good': 4, 'Excellent': 5}
train_df['General Health'] = train_df['General Health'].map(health_map)
test_df['General Health'] = test_df['General Health'].map(health_map)

# ข้อมูลเกี่ยวกับการศึกษาของผู้ป่วย
edu_map = {
    'Never attended school': 1, 'Elementary': 2, 'Some high school': 3,
    'High school graduate': 4, 'Some college or technical school': 5, 'College graduate': 6
}
train_df['Education Level'] = train_df['Education Level'].map(edu_map)
test_df['Education Level'] = test_df['Education Level'].map(edu_map)

# รายได้ 
income_map = {
    'Less than $10,000': 1, '($10,000 to less than $15,000': 2, 
    '$15,000 to less than $20,000': 3, '$20,000 to less than $25,000': 4,
    '$25,000 to less than $35,000': 5, '$35,000 to less than $50,000': 6,
    '$50,000 to less than $75,000': 7, '$75,000 or more': 8
}
train_df['Income Level'] = train_df['Income Level'].map(income_map)
test_df['Income Level'] = test_df['Income Level'].map(income_map)
# ==========================================


# 4. ใช้ LabelEncoder แปลงข้อความที่เหลือ (เช่น Male/Female, Yes/No) แค่นั้นพอ
# (เพราะคอมพิวเตอร์ข้าม 3 คอลัมน์ด้านบนให้เราเองอัตโนมัติ เนื่องจากมันกลายเป็นเลขไปแล้ว)
encode_cols = [col for col in train_df.columns if train_df[col].dtype == 'object' and col != 'ID']

le = LabelEncoder()
for col in encode_cols:
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    
    if col in test_df.columns:
        test_df[col] = test_df[col].map(lambda s: s if s in le.classes_ else le.classes_[0])
        test_df[col] = le.transform(test_df[col].astype(str))

# 5. แยก Features (X) และ Target (y)
X = train_df.drop(columns=['ID', 'History of HeartDisease or Attack'])
y = train_df['History of HeartDisease or Attack']

X_test_predict = test_df.drop(columns=['ID'])


display(X.head()) # กดรันแล้วดูที่ตารางเลยครับ คอลัมน์รายได้และการศึกษาเรียงเป็นเลข 1, 2, 3 สวยงามมาก


,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,1,1,1,40.68,1,0,0,0,0,1,0,1,1,0,4,3,64,1
1,0,0,0,24.36,1,0,0,1,0,0,1,3,0,0,6,1,50,0
2,1,1,1,27.33,0,0,0,0,0,1,1,1,1,0,4,8,61,1
3,1,0,1,27.01,0,0,0,1,0,1,0,4,0,0,3,6,74,1
4,1,1,1,34.56,1,0,0,1,0,1,1,1,1,1,3,3,98,1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. แบ่งข้อมูลเอาไว้สำหรับการสอบย่อย (Validation) เพื่อดูว่าโมเดลเราเก่งแค่ไหนก่อนจะนำไปเดาของจริง
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. สร้างและ Train โมเดลด้วยอัลกอริทึม Random Forest
print("กำลัง Train Model (อาจใช้เวลาประมาณ 10-30 วินาที ขึ้นอยู่กับขนาดข้อมูล)...")
# n_jobs=-1 คือให้คอมพิวเตอร์ใช้ CPU แบบเต็มสูบเพื่อร่นเวลาเทรน
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 3. ลองทดสอบทำนายผลกับชุด Validation ที่แบ่งไว้ 
y_pred_val = model.predict(X_val)

# พิมพ์ผลลัพธ์คะแนนความแม่นยำ
print("\n--- ผลการทดสอบโมเดล (Validation Score) ---")
print("Accuracy (ความแม่นยำ):", round(accuracy_score(y_val, y_pred_val), 4))
print(classification_report(y_val, y_pred_val, target_names=['No (0)', 'Yes (1)']))

# 4. นำโมเดลไปสอบจริง (ทำนายชุด test.csv เตรียมส่ง Kaggle)
print("\nกำลังทำนายผลลัพธ์ของไฟล์ test.csv เพื่อสร้าง submission...")
final_predictions = model.predict(X_test_predict)

# 5. นำผลลัพธ์ที่ได้ไปใส่ตารางเตรียมส่ง 
# หมายเหตุ: เพราะเราเข้า Label Encoder ไว้ (No เป็น 0, Yes เป็น 1) เราต้องแปลงกลับให้ถูกฟอร์ม Kaggle
submission_df = pd.DataFrame({
    'ID': test_df['ID'],
    'History of HeartDisease or Attack': final_predictions
})

# แปลง 0 และ 1 ให้กลับเป็นคำอธิบาย 'No' กับ 'Yes' เพื่อให้ตรงฟอร์มคะแนน  
submission_df['History of HeartDisease or Attack'] = submission_df['History of HeartDisease or Attack'].map({0: 'No', 1: 'Yes'})

# 6. บันทึก (Export) ออกมาเป็นไฟล์ csv ตรงที่เดียวกับที่คุณเซฟโปรเจ็คไว้
submission_df.to_csv('my_submission.csv', index=False)

print("✅ สำเร็จ! บันทึกไฟล์ 'my_submission.csv' เรียบร้อย 🎉")


กำลัง Train Model (อาจใช้เวลาประมาณ 10-30 วินาที ขึ้นอยู่กับขนาดข้อมูล)...

--- ผลการทดสอบโมเดล (Validation Score) ---
Accuracy (ความแม่นยำ): 0.9201
              precision    recall  f1-score   support

      No (0)       0.92      1.00      0.96     41022
     Yes (1)       0.56      0.04      0.08      3595

    accuracy                           0.92     44617
   macro avg       0.74      0.52      0.52     44617
weighted avg       0.89      0.92      0.89     44617


กำลังทำนายผลลัพธ์ของไฟล์ test.csv เพื่อสร้าง submission...
✅ สำเร็จ! บันทึกไฟล์ 'my_submission.csv' เรียบร้อย สามารถนำไฟล์นี้ไปกด upload ส่งประกวดในเว็บ Kaggle ได้เลยครับ 🎉


In [38]:
# เพิ่ม class_weight='balanced' เพื่อแก้ปัญหาคนป่วยน้อย
print("กำลัง Train Model รอบแก้ตัว (ใช้ Class Weight)...")
model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1, class_weight='balanced')
model.fit(X_train, y_train)

# ลองทดสอบทำนายผลกับชุด Validation ที่แบ่งไว้ 
y_pred_val = model.predict(X_val)

# พิมพ์ผลลัพธ์
print("\n--- ผลการทดสอบโมเดล (เน้นดูที่บรรทัด Yes) ---")
print(classification_report(y_val, y_pred_val, target_names=['No (0)', 'Yes (1)']))

# ทำนายจริง
final_predictions = model.predict(X_test_predict)

# ตรวจสอบดูว่ามันทาย Yes ไปกี่คน ทาย No ไปกี่คน 
import numpy as np
unique, counts = np.unique(final_predictions, return_counts=True)
print("\nสัดส่วนที่โมเดลทายในเทสต์เซต (0=No, 1=Yes):", dict(zip(unique, counts)))

# สร้าง Submission
submission_df = pd.DataFrame({
    'ID': test_df['ID'],
    'History of HeartDisease or Attack': final_predictions
})

# แปลงกลับเป็นตัวหนังสือ
submission_df['History of HeartDisease or Attack'] = submission_df['History of HeartDisease or Attack'].map({0: 'No', 1: 'Yes'})

# เซฟลงไฟล์ใหม่ จะได้ไม่ทับอันเก่า
submission_df.to_csv('my_submission_v2_balanced.csv', index=False)
print("✅ สำเร็จ! ซ่อมไฟล์ใหม่เป็น 'my_submission_v2_balanced.csv' ลองเอาไฟล์นี้ไปส่งดูครับ")


กำลัง Train Model รอบแก้ตัว (ใช้ Class Weight)...

--- ผลการทดสอบโมเดล (เน้นดูที่บรรทัด Yes) ---
              precision    recall  f1-score   support

      No (0)       0.97      0.79      0.87     41022
     Yes (1)       0.24      0.74      0.36      3595

    accuracy                           0.79     44617
   macro avg       0.61      0.77      0.62     44617
weighted avg       0.91      0.79      0.83     44617


สัดส่วนที่โมเดลทายในเทสต์เซต (0=No, 1=Yes): {np.int64(0): np.int64(52720), np.int64(1): np.int64(21641)}
✅ สำเร็จ! ซ่อมไฟล์ใหม่เป็น 'my_submission_v2_balanced.csv' ลองเอาไฟล์นี้ไปส่งดูครับ


In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import pandas as pd

# 1. กลยุทธ์ป้องกัน Private Leaderboard ตก: แบ่งสลับ 5 ส่วน (5-Fold CV)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ตัวแปรสำหรับเก็บผลเฉลี่ย
test_predictions_prob = np.zeros(len(X_test_predict)) # เก็บความมั่นใจ
val_scores = []
models = []

# คำนวณน้ำหนักแก้ Imbalance ให้ XGBoost (เอาจำนวนคนเป็น 'No' หารคนเป็น 'Yes')
ratio = float(np.sum(y == 0)) / np.sum(y == 1)

print("🚀 เริ่มลุยเทรน XGBoost แบบ 5-Fold Cross Validation...")

# วนลูป 5 รอบ (สลับส่วนเทสและเทรนไปเรื่อยๆ)
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # สร้าง XGBoost
    model_xgb = xgb.XGBClassifier(
    n_estimators=800,
    learning_rate=0.01,
    max_depth=5,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1
)

    
    # เริ่มเทรนตัวที่ 1... 5
    model_xgb.fit(X_tr, y_tr)
    
    # ทดสอบความแม่นยำในกลุ่มที่แบ่งไว้รอบนั้นๆ
    val_pred = model_xgb.predict(X_va)
    score = accuracy_score(y_va, val_pred) 
    val_scores.append(score)
    print(f"✅ สร้างโมเดลของวิชา (Fold) {fold+1}/5 สำเร็จ | ความแม่นยำ: {score:.4f}")
    
    # [จุดสำคัญ] ให้โมเดลลองไปสอบจริง (Test) แล้วบอกมาเป็นเปอร์เซ็นต์ความมั่นใจ
    # แล้วเอาค่าสะสมมาบวกกัน 5 รอบ หาร 5 เป็นค่าเฉลี่ยทั้งหมด
    test_predictions_prob += model_xgb.predict_proba(X_test_predict)[:, 1] / kf.n_splits

print(f"\n🌟 ทำการเทรนเสร็จสมบูรณ์! ยืนยันความแม่นยำเฉลี่ยที่: {np.mean(val_scores):.4f}")

# 2. ปรับตัวเลขความน่าจะเป็นให้กลายเป็นการฟันธง
# ถ้า 5 ตัวโหวตเฉลี่ยแล้วมั่นใจเกินครึ่ง (>= 0.5) ให้ตัดสินว่า 1(Yes) ไม่งั้น 0(No)
final_xgb_classes = (test_predictions_prob >= 0.5).astype(int)

# ----------------- สร้างไฟล์ส่ง -----------------
submission_df_xgb = pd.DataFrame({
    'ID': test_df['ID'],
    'History of HeartDisease or Attack': final_xgb_classes
})

# แปลง 0/1 กลับเป็นข้อความ 'No' / 'Yes'
submission_df_xgb['History of HeartDisease or Attack'] = submission_df_xgb['History of HeartDisease or Attack'].map({0: 'No', 1: 'Yes'})

submission_df_xgb.to_csv('my_submission_v3_CV_xgboost.csv', index=False)
print("🎉 ไปครับ! นำ 'my_submission_v3_CV_xgboost.csv' ไปส่งได้เลย โอกาสคะแนนขึ้นและเกาะ Private แน่นๆ มีสูงมากครับ")


🚀 เริ่มลุยเทรน XGBoost แบบ 5-Fold Cross Validation...
✅ สร้างโมเดลของวิชา (Fold) 1/5 สำเร็จ | ความแม่นยำ: 0.7598
✅ สร้างโมเดลของวิชา (Fold) 2/5 สำเร็จ | ความแม่นยำ: 0.7564
✅ สร้างโมเดลของวิชา (Fold) 3/5 สำเร็จ | ความแม่นยำ: 0.7579
✅ สร้างโมเดลของวิชา (Fold) 4/5 สำเร็จ | ความแม่นยำ: 0.7592
✅ สร้างโมเดลของวิชา (Fold) 5/5 สำเร็จ | ความแม่นยำ: 0.7610

🌟 ทำการเทรนเสร็จสมบูรณ์! ยืนยันความแม่นยำเฉลี่ยที่: 0.7589
🎉 ไปครับ! นำ 'my_submission_v3_CV_xgboost.csv' ไปส่งได้เลย โอกาสคะแนนขึ้นและเกาะ Private แน่นๆ มีสูงมากครับ


In [41]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

# ==========================================
# 🏆 V4: XGBoost + LightGBM Ensemble + Threshold Tuning
# ==========================================

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ratio = float(np.sum(y == 0)) / np.sum(y == 1)

# เก็บค่า probability จากทั้ง 2 โมเดล
xgb_test_proba = np.zeros(len(X_test_predict))
lgb_test_proba = np.zeros(len(X_test_predict))

# เก็บ probability ของ Validation เพื่อหา threshold ที่ดีที่สุด
oof_proba = np.zeros(len(X))  # Out-of-Fold predictions

print("🚀 เริ่มเทรน 2 โมเดลพร้อมกัน (XGBoost + LightGBM) แบบ 5-Fold...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # --- โมเดลที่ 1: XGBoost (จูนแล้ว) ---
    model_xgb = xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.02,
        max_depth=5,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=ratio,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    model_xgb.fit(X_tr, y_tr)

    # --- โมเดลที่ 2: LightGBM ---
    model_lgb = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.02,
        max_depth=5,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=ratio,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    model_lgb.fit(X_tr, y_tr)

    # รวมความมั่นใจ 2 โมเดลเข้าด้วยกัน (Ensemble) สำหรับ Validation
    xgb_val_proba = model_xgb.predict_proba(X_va)[:, 1]
    lgb_val_proba = model_lgb.predict_proba(X_va)[:, 1]
    oof_proba[val_idx] = (xgb_val_proba + lgb_val_proba) / 2

    # สะสมผลทำนาย Test 
    xgb_test_proba += model_xgb.predict_proba(X_test_predict)[:, 1] / kf.n_splits
    lgb_test_proba += model_lgb.predict_proba(X_test_predict)[:, 1] / kf.n_splits

    # แสดงคะแนนแต่ละ fold
    fold_f1 = f1_score(y_va, (oof_proba[val_idx] >= 0.5).astype(int))
    print(f"✅ Fold {fold+1}/5 | F1-Score (ensemble): {fold_f1:.4f}")

# รวมผลทำนาย Test จาก 2 โมเดล
ensemble_test_proba = (xgb_test_proba + lgb_test_proba) / 2

# ==========================================
# 🎯 ค้นหา Threshold ที่ดีที่สุดจากข้อมูลจริง (ไม่ได้เดามัวๆ)
# ==========================================
print("\n🔍 กำลังค้นหา Threshold ที่ดีที่สุด...")
best_threshold = 0.5
best_f1 = 0

for th in np.arange(0.30, 0.70, 0.01):
    pred = (oof_proba >= th).astype(int)
    score = f1_score(y, pred)
    if score > best_f1:
        best_f1 = score
        best_threshold = th

print(f"🏅 Threshold ที่ดีที่สุดคือ: {best_threshold:.2f} (F1 บน CV = {best_f1:.4f})")

# ตัดสินใจ Final Prediction ด้วย threshold ที่ดีที่สุด
final_classes = (ensemble_test_proba >= best_threshold).astype(int)

# ดูสัดส่วนที่ทาย
unique, counts = np.unique(final_classes, return_counts=True)
print(f"สัดส่วนที่ทาย: {dict(zip(['No','Yes'], counts))}")

# สร้างไฟล์ Submission
submission_df = pd.DataFrame({
    'ID': test_df['ID'],
    'History of HeartDisease or Attack': final_classes
})
submission_df['History of HeartDisease or Attack'] = submission_df['History of HeartDisease or Attack'].map({0: 'No', 1: 'Yes'})

submission_df.to_csv('my_submission_v4_ensemble.csv', index=False)
print("\n🎉 สำเร็จ! นำไฟล์ 'my_submission_v4_ensemble.csv' ไปส่ง Kaggle ได้เลยครับ!")


🚀 เริ่มเทรน 2 โมเดลพร้อมกัน (XGBoost + LightGBM) แบบ 5-Fold...
✅ Fold 1/5 | F1-Score (ensemble): 0.3547
✅ Fold 2/5 | F1-Score (ensemble): 0.3446
✅ Fold 3/5 | F1-Score (ensemble): 0.3498
✅ Fold 4/5 | F1-Score (ensemble): 0.3521
✅ Fold 5/5 | F1-Score (ensemble): 0.3525

🔍 กำลังค้นหา Threshold ที่ดีที่สุด...
🏅 Threshold ที่ดีที่สุดคือ: 0.69 (F1 บน CV = 0.4056)
สัดส่วนที่ทาย: {'No': np.int64(59628), 'Yes': np.int64(14733)}

🎉 สำเร็จ! นำไฟล์ 'my_submission_v4_ensemble.csv' ไปส่ง Kaggle ได้เลยครับ!
